# Phase 4: Exploratory Data Analysis (EDA)
## Flight Delays Are Contagious: Predicting Delay Propagation Across Consecutive Aircraft Operations

**Research Objective:** Investigate the extent to which delays from preceding flight operations predict delays in subsequent flights operated by the same aircraft.

**Dataset:** `data/processed/flight_rotation_pairs.parquet`  
**Sample Size:** 6,768,487 validated consecutive flight pairs  
**Time Horizon:** Full Year 2025 (BTS On-Time Performance)  
**Targets:** Classification (`NEXT_ARR_DEL15`), Regression (`NEXT_ARR_DELAY`)


In [ ]:
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure writable matplotlib cache
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
plt.style.use("seaborn-v0_8-whitegrid")

# Connect to DuckDB
con = duckdb.connect()
DATA_PATH = "data/processed/flight_rotation_pairs.parquet"
print(f"DuckDB version: {duckdb.__version__}")
print(f"Connected to dataset: {DATA_PATH}")


---
## 1. Target Variable Analysis

We investigate the distributional characteristics of both prediction targets:
1. **Binary Classification Target (`NEXT_ARR_DEL15`):** Standard FAA/BTS definition of flight arrival delay (1 if arrival delay $\ge 15$ minutes, 0 otherwise).
2. **Continuous Regression Target (`NEXT_ARR_DELAY`):** Difference between actual gate arrival and scheduled gate arrival in minutes (negative values indicate early arrivals).


In [ ]:
# Query target summary metrics from Parquet
q_targets = f"""
SELECT 
    COUNT(*) AS total_pairs,
    SUM(NEXT_ARR_DEL15) AS count_del15,
    SUM(CASE WHEN NEXT_ARR_DEL15 = 0 THEN 1 ELSE 0 END) AS count_ontime,
    ROUND(AVG(NEXT_ARR_DEL15) * 100.0, 2) AS pct_del15,
    ROUND(AVG(NEXT_ARR_DELAY), 2) AS mean_arr_delay,
    ROUND(MEDIAN(NEXT_ARR_DELAY), 2) AS median_arr_delay,
    ROUND(STDDEV(NEXT_ARR_DELAY), 2) AS std_arr_delay,
    ROUND(MIN(NEXT_ARR_DELAY), 2) AS min_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.25), 2) AS q25_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.75), 2) AS q75_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.90), 2) AS p90_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.99), 2) AS p99_arr_delay,
    ROUND(MAX(NEXT_ARR_DELAY), 2) AS max_arr_delay
FROM read_parquet('{DATA_PATH}')
"""
df_targets = con.execute(q_targets).df()
display(df_targets)

# Display publication figure
display(Image(filename="results/figures/fig01_target_distributions.png"))


### Analytical Interpretation: Target Distributions & Skewness

1. **Class Imbalance (Classification):**
   - The overall delay rate is **22.02%** ($1,490,586$ delayed vs $5,277,901$ on-time flights), representing an imbalance ratio of **3.54:1**.
   - While moderate, this imbalance requires threshold-independent evaluation metrics (such as PR-AUC / Average Precision, ROC-AUC, and F1-score) rather than raw accuracy.
2. **Severe Right Skewness (Regression):**
   - The continuous delay distribution exhibits classic heavy-tailed, asymmetric right skew:
     - **Median Arrival Delay:** $-6.00$ minutes (over 60% of flights arrive early due to scheduled block-time padding).
     - **Mean Arrival Delay:** $+7.78$ minutes (pulled upward by positive outliers).
     - **Interquartile Range (IQR):** $[-15.0, +11.0]$ minutes (a 26-minute central spread).
     - **90th Percentile (P90):** $+47.0$ minutes | **99th Percentile (P99):** $+214.0$ minutes | **Maximum:** $+4,336.0$ minutes.
3. **Implications for Machine Learning:**
   - Standard Mean Squared Error (MSE) loss is hypersensitive to extreme multi-hour delays.
   - Robust evaluation metrics such as **Mean Absolute Error (MAE)** and **Median Absolute Error (MedAE)** must be prioritized in model evaluation.


---
## 2. Previous-Flight Delay Propagation Dynamics

**Analytical Question:** How strongly does an aircraft's preceding arrival delay condition the likelihood and severity of delay on its next scheduled flight?


In [ ]:
# Display delay propagation across prior delay intervals
df_bins = pd.read_csv("results/tables/eda_delay_propagation_bins.csv")
display(df_bins)

# Display transition matrix
df_trans = pd.read_csv("results/tables/eda_transition_matrix.csv")
display(df_trans)

# Display publication figure
display(Image(filename="results/figures/fig02_delay_propagation_relationships.png"))


### Analytical Interpretation: Delay Propagation Dynamics

1. **Empirical Delay Response Across Intervals:**
   - When the preceding flight is early ($<0$ min), subsequent flight delay is only **12.43%** ($511,189 / 4,112,539$) with a negative mean delay ($-2.05$ min).
   - Once preceding delay enters the **$30–44$ minute window**, the subsequent delay rate crosses the 50% threshold (**50.84%**, $140,826 / 276,984$).
   - For severe delay categories ($>60$ min), subsequent delay probability stabilizes around **$60–62\%$**, while the *magnitude* of delay escalates (mean delay reaches $+92.56$ min for preceding delays $\ge 120$ min).
2. **Transition Probability, Relative Risk & Odds Ratio with 95% CIs:**
   - $P(\text{Next Delayed} \mid \text{Prev Delayed}) = 755,257 / 1,504,840 = \mathbf{50.19\%}$
   - $P(\text{Next Delayed} \mid \text{Prev On-Time}) = 735,329 / 5,263,647 = \mathbf{13.97\%}$
   - **Relative Risk ($RR$):** $\mathbf{3.5926}$ (95% CI: $[3.5831, 3.6021]$)
   - **Odds Ratio ($OR$):** $\mathbf{6.2048}$ (95% CI: $[6.1798, 6.2299]$)
   - Preceding flight delay is associated with a statistically significant, three-and-a-half-fold increase in the relative risk of subsequent flight delay.


---
## 3. Turnaround Buffer Capacity & Deficit Dynamics

**Analytical Question:** How does scheduled ground turnaround time act as an operational buffer, and what outcomes are observed when incoming delay exceeds this buffer?


In [ ]:
# Display turnaround buffer bins
df_buf = pd.read_csv("results/tables/eda_turnaround_buffer_bins.csv")
display(df_buf)

# Display turnaround deficit impact
df_deficit = pd.read_csv("results/tables/eda_turnaround_deficit_impact.csv")
display(df_deficit)

# Display publication figure
display(Image(filename="results/figures/fig03_turnaround_buffer_dynamics.png"))


### Analytical Interpretation: Turnaround Buffer & Deficit Observations

1. **Buffer Capacity and Subsequent Delay Rates:**
   - In tightly scheduled turns ($<45$ min buffer), delay propagation is observed in **84.01%** ($193,201 / 229,986$) of cases where the preceding flight was delayed.
   - For standard turns ($60–74$ min), the rate is **64.48%** ($138,479 / 214,754$).
   - For extended layovers ($\ge 180$ min), the rate falls to **19.24%** ($106,317 / 552,512$), which is below the unconditional fleet average.
2. **Turnaround Deficit Group Performance:**
   - In the turnaround deficit cohort ($N = 382,937$, $5.66\%$ of pairs), the subsequent delay rate is **99.50%** ($381,015 / 382,937$) with a mean delay of $+121.86$ minutes.
   - In the non-deficit cohort ($N = 6,385,550$, $94.34\%$ of pairs), the subsequent delay rate is **17.38%** ($1,109,571 / 6,385,550$) with a mean delay of $+0.93$ minutes.
   - **Descriptive Framing:** A turnaround deficit indicates that incoming arrival occurred after scheduled pushback. Because external factors (adverse weather, airport ground delays) often co-occur with major delays, this result represents a strong descriptive association rather than isolated causal proof.


---
## 4. Temporal, Diurnal, and Seasonal Delay Patterns

**Analytical Question:** How do subsequent delay rates and propagation frequencies vary across departure hours, days of the week, and months?


In [ ]:
# Display hourly diurnal patterns with conditional distinctions
df_hour = pd.read_csv("results/tables/eda_hourly_patterns.csv")
display(df_hour.head(10))

# Display publication figure
display(Image(filename="results/figures/fig04_temporal_delay_patterns.png"))


### Analytical Interpretation: Diurnal Variation & Seasonality

1. **Hourly Conditional Variation:**
   - For early morning departures (05:00), $P(\text{Next Del} \mid \text{Prev Del}) = 11.14\%$ ($6,878 / 61,764$), while $P(\text{Next Del} \mid \text{Prev On-Time}) = 7.33\%$ ($9,541 / 130,136$).
   - By 10:00, $P(\text{Next Del} \mid \text{Prev Del})$ reaches $51.76\%$ ($30,218 / 58,381$), while on-time predecessors yield only $11.35\%$ ($41,318 / 363,881$).
   - At the 17:00 peak, $P(\text{Next Del} \mid \text{Prev Del})$ reaches $71.23\%$ ($71,813 / 100,823$), whereas $P(\text{Next Del} \mid \text{Prev On-Time})$ is $18.37\%$ ($59,414 / 323,514$).
2. **Weekly & Monthly Patterns:**
   - Unconditional delay rates are highest in June ($25.8\%$) and July ($26.4\%$) during convective summer weather, and lowest in autumn ($17.5–19.2\%$) during shoulder travel periods.
   - Across weekdays, Friday exhibits the highest delay frequency ($25.1\%$) and Tuesday/Saturday the lowest ($19.2–19.4\%$).


---
## 5. Operating Carrier & Airport Operational Heterogeneity

**Analytical Question:** How do delay propagation patterns vary across major airline operators and turnaround hub airports?


In [ ]:
# Display top operating carriers
df_carrier = pd.read_csv("results/tables/eda_carrier_summary.csv")
display(df_carrier.head(10))

# Display top turnaround airports
df_ap = pd.read_csv("results/tables/eda_turnaround_airport_summary.csv")
display(df_ap.head(10))

# Display publication figure
display(Image(filename="results/figures/fig05_carrier_airport_heterogeneity.png"))


### Analytical Interpretation: Carrier & Airport Heterogeneity

1. **Carrier Operational Differences:**
   - Network legacy carriers (Delta, United) exhibit lower overall deficit rates ($\sim 3.7–3.8\%$) and lower delay propagation frequencies ($40.2–46.9\%$), reflecting network route structures, hub spare aircraft availability, and block-time padding.
   - Point-to-point and regional carriers (Southwest, SkyWest, PSA) operate tighter ground turnarounds with propagation rates between $50.9\%$ and $59.1\%$.
   - When a turnaround deficit occurs, all carriers experience delay rates exceeding $98.4\%$, showing that buffer exhaustion is accompanied by high delay rates regardless of carrier brand.
2. **Turnaround Hub Differences:**
   - Densely scheduled hubs with runway constraints (ORD, DFW, DEN, CLT, LGA, DCA) exhibit propagation rates exceeding $50–55\%$.
   - High-capacity western hubs (SLC, PHX, LAX) exhibit lower propagation rates ($40–47\%$) and lower overall delay rates.


---
## 6. Aircraft Fleet Patterns and Delay Concentration

**Analytical Question:** How are positive delay minutes distributed across the commercial aircraft fleet?


In [ ]:
# Display consecutive delay progression
df_consec = pd.read_csv("results/tables/eda_aircraft_consecutive_delays.csv")
display(df_consec)

# Display fleet concentration metrics
df_conc = pd.read_csv("results/tables/eda_aircraft_concentration.csv")
display(df_conc)

# Display publication figure
display(Image(filename="results/figures/fig06_aircraft_level_patterns.png"))


### Analytical Interpretation: Aircraft Fleet Concentration & Gini Audit

1. **Consecutive Delay History:**
   - $0$ prior delays today: Subsequent delay rate is **14.26%** ($822,567 / 5,768,036$), mean delay $-0.13$ min.
   - $1$ prior delay today: Subsequent delay rate is **62.54%** ($383,309 / 612,913$), mean delay $+47.44$ min.
   - $2$ prior delays today: Subsequent delay rate is **71.43%** ($174,285 / 244,011$), mean delay $+58.61$ min.
   - $3+$ prior delays today: Subsequent delay rate is **76.94%** ($110,425 / 143,527$), mean delay $+69.63$ min.
2. **Audited Gini Coefficient on Positive Delay Minutes & Exact Population:**
   - **Population Scope:** Strictly covers **6,124 unique commercial aircraft** across the **6,768,487 completed outbound flight legs ($f_{\text{next}}$)** in `flight_rotation_pairs.parquet` (not all completed 2025 flights).
   - **Aggregation Formula:** $\text{POSITIVE\_DELAY\_MINUTES}_i = \sum_{j=1}^{M_i} \max(0, \text{NEXT\_ARR\_DELAY}_{i,j})$ where $\sum M_i = 6,768,487$.
   - **Total Positive Delay:** **112,174,297 minutes** ($1,869,571.6$ aircraft-hours).
   - **Gini Coefficient ($G$):** **0.2890** ($0.289047$).
   - **Fleet Concentration Shares:** Top 10%: **20.08%**, Top 20%: **34.67%**, Top 50%: **70.00%**.
   - **Revised Methodological Interpretation:** This indicates a **moderate concentration of total positive delay burden** across the fleet. **Critical Limitation:** Because this aggregate delay burden metric is **not normalized by aircraft flight volume or operating hours**, it **must not be claimed to prove that delays are not concentrated among persistently unreliable aircraft** (airframes with higher flight volume naturally accumulate more total delay minutes).


---
## 7. Feature Relationships, Correlations, and Multicollinearity

**Analytical Question:** Which candidate predictors exhibit the strongest relationships with subsequent delay, and which present collinearity risks?


In [ ]:
# Display correlation table
df_corr = pd.read_csv("results/tables/eda_correlation_matrix.csv")
display(df_corr.sort_values(by="spearman_with_NEXT_ARR_DELAY", ascending=False))

# Display feature redundancy audit
df_red = pd.read_csv("results/tables/eda_feature_redundancy_audit.csv")
display(df_red)

# Display publication figure
display(Image(filename="results/figures/fig07_feature_correlations.png"))


### Analytical Interpretation: Correlation & Predictive Structure

1. **Top Monotonic Predictors (Spearman $\rho$):**
   - `PREV_AIRCRAFT_CONSECUTIVE_DEL15` ($\rho = +0.4085$)
   - `IS_TURNAROUND_DEFICIT` ($r = +0.4909$, $\rho = +0.3779$)
   - `ESTIMATED_REMAINING_TURNAROUND_MINUTES` ($\rho = -0.3387$)
   - `PREV_ARR_DEL15` ($\rho = +0.3252$)
   - `PREV_ARR_DELAY` ($\rho = +0.3228$)
   - `NEXT_CRS_DEP_HOUR` ($\rho = +0.1778$)
2. **Multicollinearity Findings:**
   - `NEXT_DISTANCE` and `NEXT_CRS_ELAPSED_TIME` ($r = 0.981$) are virtually identical proxies for flight block time.
   - `PREV_ARR_DELAY` and `PREV_DEP_DELAY` ($r = 0.884$) are highly collinear; `PREV_ARR_DELAY` is preferred as it conditions gate arrival.
   - Non-linear rank correlations (Spearman) are substantially stronger than linear correlations (Pearson) for remaining buffer, demonstrating the presence of non-linear threshold effects.


---
## 8. Permitted Syllabus Model Recommendations & Phase 5 Handoff

Based on the empirical findings, the following model families from the **permitted project syllabus** are recommended for comparative evaluation:

### Permitted Model Families
1. **Linear and Generalized Linear Models:**
   - **Logistic Regression (L1/L2 regularized):** Interpretable classification benchmark providing odds ratio verification.
   - **Linear and Multiple Regression / Ridge / Lasso:** Baseline continuous regression models; requires feature transformation to handle target skewness.
   - **Polynomial Regression:** Evaluates non-linear buffer threshold effects within a linear parameter framework.
2. **Decision Trees and Tree Ensembles:**
   - **Decision Trees (CART - Classifier & Regressor):** Capable of learning the sharp non-linear split at $\text{buffer} \le 0$.
   - **Random Forest (Classifier & Regressor):** Non-linear ensemble capable of modeling feature interactions without severe overfitting.
3. **Kernel Methods (Classification Only):**
   - **Support Vector Classifier (Linear / RBF SVC):** Evaluates non-linear decision boundaries for classification on standardized subsets. *(SVR is strictly excluded as it is out-of-syllabus)*.
4. **Neural Network Architectures:**
   - **Feedforward Neural Networks (Multi-Layer Perceptron / MLPClassifier & MLPRegressor):** Approximates complex non-linear delay accumulation surfaces.

### Out-of-Syllabus Exclusions
- **Gradient Boosted Decision Trees:** XGBoost, LightGBM, and CatBoost are strictly excluded.
- **Support Vector Regression (SVR):** Strictly excluded from continuous regression tasks due to syllabus alignment.

### Methodological Guardrails for Phase 5 & 6
- **Model Evaluation:** No single model family is assumed to be superior prior to rigorous comparative empirical evaluation.
- **Chronological Splitting:** Random train/test splits are prohibited; temporal splits must be enforced to respect causality.
- **Robust Metrics:** Evaluation must report MAE, MedAE, PR-AUC, and ROC-AUC alongside standard metrics.
